# SpecFreak — Phase 2: TF-IDF + Cosine Similarity
### NLP-Based Game Recommendation Engine
**What this phase does:**
- Combines game description + genre + platform + type into one text feature
- Converts that text into numbers using TF-IDF
- Takes your natural language prompt and finds the most similar games using Cosine Similarity
- Returns top N game recommendations

## Step 1 — Install & Import Libraries

In [ ]:
# Run this cell first to make sure all libraries are available
# In Google Colab these are already pre-installed, so this should run fine

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2 — Load the Dataset

In [ ]:
# ---------------------------------------------------------
# IMPORTANT: Make sure your CSV file is uploaded to Colab
# Go to the Files panel on the left → Upload
# Upload: games_of_all_time_cleaned_normalized.csv
# ---------------------------------------------------------

df = pd.read_csv('games_of_all_time_cleaned_normalized.csv')

print(f"Dataset loaded: {df.shape[0]} games, {df.shape[1]} columns")
print("\nColumns available:")
print(df.columns.tolist())
print("\nFirst 3 games:")
df[['game_name', 'genre', 'platform', 'type', 'meta_score']].head(3)

Dataset loaded: 8831 games, 12 columns

Columns available:
['game_name', 'meta_score', 'user_score', 'platform', 'description', 'url', 'developer', 'genre', 'type', 'rating', 'meta_score_norm_0_1', 'user_score_norm_0_1']

First 3 games:


,game_name,genre,platform,type,meta_score
0,The Legend of Zelda: Ocarina of Time,"['Action Adventure', 'Fantasy']",['nintendo-64'],singleplayer,99.0
1,Super Mario Galaxy,"['Action', 'Platformer', '3D']",['wii'],singleplayer,97.0
2,Super Mario Galaxy 2,"['Action', 'Platformer', '3D']",['wii'],singleplayer,97.0


## Step 3 — Feature Engineering (Combine Text Columns)

In [ ]:
# ------------------------------------------------------------------
# WHAT WE ARE DOING HERE:
# TF-IDF works on a single text column.
# So we COMBINE: description + genre + platform + type
# into one big text string called 'combined_features'.
# This gives the model more context about each game.
# ------------------------------------------------------------------

def clean_list_string(text):
    """
    Genre and platform are stored as strings like:
    "['Action Adventure', 'Fantasy']"
    This function converts that into plain text:
    "Action Adventure Fantasy"
    """
    if pd.isna(text):
        return ''
    # Remove brackets, quotes, commas
    text = re.sub(r"[\[\]'\"]", '', str(text))
    text = re.sub(r',', ' ', text)
    return text.strip().lower()

def clean_text(text):
    """Basic text cleaning for description and type fields."""
    if pd.isna(text):
        return ''
    return str(text).strip().lower()


# Apply cleaning
df['clean_genre']    = df['genre'].apply(clean_list_string)
df['clean_platform'] = df['platform'].apply(clean_list_string)
df['clean_type']     = df['type'].apply(clean_text)
df['clean_desc']     = df['description'].apply(clean_text)

# Combine all into ONE text field
# We repeat genre and type 2x to give them more weight in TF-IDF
df['combined_features'] = (
    df['clean_desc'] + ' ' +
    df['clean_genre'] + ' ' + df['clean_genre'] + ' ' +   # genre repeated for emphasis
    df['clean_platform'] + ' ' +
    df['clean_type'] + ' ' + df['clean_type']              # type repeated for emphasis
)

print("Feature engineering done!")
print("\nSample combined feature for game 1:")
print(df['combined_features'].iloc[0][:300], '...')

Feature engineering done!

Sample combined feature for game 1:
as a young boy, link is tricked by ganondorf, the king of the gerudo thieves. the evil human uses link to gain access to the sacred realm, where he places his tainted hands on triforce and transforms the beautiful hyrulean landscape into a barren wasteland. link is determined to fix the problems he  ...


## Step 4 — Build the TF-IDF Matrix

In [ ]:
# ------------------------------------------------------------------
# WHAT IS TF-IDF?
# TF  = Term Frequency   → how often a word appears in THIS game's text
# IDF = Inverse Document Frequency → how rare that word is across ALL games
# TF-IDF score = high if the word is COMMON in this game but RARE overall
# Result: Each game becomes a vector of numbers (one number per unique word)
# ------------------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=10000,    # use top 10,000 most important words
    stop_words='english',  # remove common words like 'the', 'is', 'and'
    ngram_range=(1, 2),    # consider single words AND 2-word phrases (e.g. 'open world')
    min_df=2               # ignore words that appear in less than 2 games
)

print("Building TF-IDF matrix... (this may take 10-20 seconds)")
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(f"\nTF-IDF matrix built successfully!")
print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} games")
print(f"  → {tfidf_matrix.shape[1]} unique words/phrases")

Building TF-IDF matrix... (this may take 10-20 seconds)

TF-IDF matrix built successfully!
Matrix shape: (8831, 10000)
  → 8831 games
  → 10000 unique words/phrases


## Step 5 — The Recommendation Function

In [ ]:
# ------------------------------------------------------------------
# WHAT IS COSINE SIMILARITY?
# It measures the angle between two vectors.
# Score = 1.0  → exact same direction = very similar
# Score = 0.0  → completely different
#
# HOW WE USE IT:
# 1. Take the user's text prompt
# 2. Convert it to a TF-IDF vector using the SAME vectorizer
# 3. Compare it against ALL 8831 game vectors
# 4. Return games with highest similarity score
# ------------------------------------------------------------------

def recommend_games(user_prompt, top_n=10):
    """
    Takes a natural language prompt from the user
    and returns the top_n most similar games.

    Parameters:
        user_prompt (str) : e.g. "dark story driven RPG with open world"
        top_n (int)       : how many recommendations to return (default 10)

    Returns:
        DataFrame with recommended games and their similarity scores
    """

    # Step A: Convert user prompt to TF-IDF vector
    user_vector = tfidf.transform([user_prompt.lower()])

    # Step B: Compute cosine similarity between user vector and ALL games
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

    # Step C: Get indices of top N most similar games
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    # Step D: Build results dataframe
    results = df.iloc[top_indices][[
        'game_name', 'genre', 'platform', 'type',
        'meta_score', 'user_score', 'rating'
    ]].copy()

    results['similarity_score'] = similarity_scores[top_indices].round(4)
    results = results.reset_index(drop=True)
    results.index += 1  # Start ranking from 1

    return results


print("Recommendation function is ready!")
print("Now go to the next cell and test it.")

Recommendation function is ready!
Now go to the next cell and test it.


## Step 6 — Test the Recommender ✅

In [ ]:
# ------------------------------------------------------------------
# CHANGE THE PROMPT BELOW TO ANYTHING YOU WANT AND RUN THIS CELL
# ------------------------------------------------------------------

user_prompt = "dark story driven RPG with open world exploration and rich narrative"

print(f"User Prompt: '{user_prompt}'")
print("=" * 70)

results = recommend_games(user_prompt, top_n=10)
print(results.to_string())

User Prompt: 'dark story driven RPG with open world exploration and rich narrative'
                                      game_name                                                                      genre                                       platform          type  meta_score  user_score rating  similarity_score
1                       Kena: Bridge of Spirits                              ['Action Adventure', 'General', 'Open-World']                        ['pc', 'playstation-5']  singleplayer       81.50        81.5      T            0.3501
2                            Shape of the World                                         ['Action Adventure', 'Open-World']                              ['playstation-4']  singleplayer       72.00        72.0      E            0.2664
3   The Witcher 3: Wild Hunt - Complete Edition                                             ['Role-Playing', 'Action RPG']                                     ['switch']  singleplayer       85.00        85.0      M   

In [ ]:
# Test with a different prompt
user_prompt2 = "old school platformer with coins and jumping"

print(f"User Prompt: '{user_prompt2}'")
print("=" * 70)

results2 = recommend_games(user_prompt2, top_n=10)
print(results2.to_string())

User Prompt: 'old school platformer with coins and jumping'
                                    game_name                                      genre                       platform          type  meta_score  user_score rating  similarity_score
1                             Mega Coin Squad             ['Action', 'Platformer', '2D']                   ['xbox-one']  singleplayer        74.0        64.0      T            0.2643
2                                 JumpJet Rex             ['Action', 'Platformer', '2D']                         ['pc']  singleplayer        76.0        70.0      T            0.2533
3                                   SwapQuest                      ['Action', 'General']           ['playstation-vita']  singleplayer        64.0        70.0   E10+            0.2488
4                       Slain: Back from Hell             ['Action', 'Platformer', '2D']                         ['pc']  singleplayer        74.0        81.0      T            0.2442
5                        

In [ ]:
# Test with another prompt
user_prompt3 = "relaxing puzzle game suitable for kids with cute graphics"

print(f"User Prompt: '{user_prompt3}'")
print("=" * 70)

results3 = recommend_games(user_prompt3, top_n=10)
print(results3.to_string())

User Prompt: 'relaxing puzzle game suitable for kids with cute graphics'
                         game_name                                   genre           platform          type  meta_score  user_score rating  similarity_score
1                 Puzzle Dimension  ['Miscellaneous', 'Puzzle', 'General']  ['playstation-3']  singleplayer        79.0        79.0      E            0.2377
2                      Pony Island                   ['Puzzle', 'General']             ['pc']  singleplayer        86.0        74.0      T            0.2101
3                         Maquette                   ['Puzzle', 'General']  ['playstation-5']  singleplayer        70.0        63.0      T            0.2099
4                             RUSH  ['Miscellaneous', 'Puzzle', 'General']          ['wii-u']  singleplayer        77.0        75.0      E            0.2083
5                     Room to Grow                   ['Puzzle', 'General']             ['pc']  singleplayer        76.0        76.0      T    

## Step 7 — Interactive Mode (Type Your Own Prompt)

In [ ]:
# ------------------------------------------------------------------
# Run this cell to enter your own prompt live in Colab
# ------------------------------------------------------------------

while True:
    prompt = input("\nDescribe the game you want (or type 'quit' to stop): ")

    if prompt.lower() == 'quit':
        print("Exiting recommendation mode.")
        break

    if len(prompt.strip()) < 3:
        print("Please enter a longer description.")
        continue

    print(f"\nTop 10 games for: '{prompt}'")
    print("-" * 60)
    res = recommend_games(prompt, top_n=20)
    print(res[['game_name', 'genre', 'type', 'meta_score', 'similarity_score']].to_string())


Describe the game you want (or type 'quit' to stop): old school platformer with coins and jumping

Top 10 games for: 'old school platformer with coins and jumping'
------------------------------------------------------------
                                    game_name                                                          genre          type  meta_score  similarity_score
1                             Mega Coin Squad                                 ['Action', 'Platformer', '2D']  singleplayer       74.00            0.3204
2   The Witcher 3: Wild Hunt - Blood and Wine                                 ['Role-Playing', 'Action RPG']  singleplayer       91.50            0.1961
3                                   SwapQuest                                          ['Action', 'General']  singleplayer       64.00            0.1949
4            Danganronpa: Trigger Happy Havoc                                  ['Adventure', 'Visual Novel']  singleplayer       81.00            0.1850
5        

## Phase 2 Complete ✅

**What was built in this phase:**

| Step | What happened |
|------|---------------|
| Feature Engineering | Combined description + genre + platform + type into one text |
| TF-IDF Vectorizer | Converted 8,831 game texts into numerical vectors (10,000 features) |
| Cosine Similarity | Measures how close a user prompt is to each game vector |
| Recommender Function | Returns top N games ranked by similarity score |

**Next → Phase 3:** Add sentiment scoring using meta_score_norm_0_1 + user_score_norm_0_1 to improve ranking quality.

## Phase 3 — Sentiment + Tags + Hybrid Ranking
**Changes in this version:**
- Results increased from 10 → **20**
- New **tags column** merged from Steam dataset
- Tags also used in TF-IDF for better matching
- Keyword expansion + normalized scoring from previous fix


### Step 1 — Compute Sentiment Score

In [ ]:
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Must run Phase 2 cells first (Cells 3-11) before this
df['meta_score'] = pd.to_numeric(df['meta_score'], errors='coerce')
df['user_score']  = pd.to_numeric(df['user_score'],  errors='coerce')

def compute_sentiment(meta, user):
    if pd.notna(meta) and pd.notna(user): return 0.6*(meta/100)+0.4*(user/10)
    elif pd.notna(meta): return meta/100
    elif pd.notna(user): return user/10
    else: return 0.5

df['sentiment_score'] = df.apply(
    lambda r: compute_sentiment(r['meta_score'], r['user_score']), axis=1
)
print('Sentiment done. Range:', df['sentiment_score'].min().round(3), '-', df['sentiment_score'].max().round(3))


### Step 2 — Load & Merge Steam Tags
**Before running this cell:**
1. Go to `kaggle.com/datasets/trolukovich/steam-games-complete-dataset`
2. Download → upload `steam_games.csv` to this Colab session
3. Then run this cell


In [ ]:
# ---------------------------------------------------------------
# MERGE STEAM TAGS INTO YOUR DATASET
# Matches games by name (case-insensitive, fuzzy-stripped)
# Games not found in Steam dataset get tags = 'No tags found'
# ---------------------------------------------------------------

def normalize_name(name):
    """Lowercase, strip punctuation for fuzzy name matching."""
    name = str(name).lower().strip()
    name = re.sub(r"[^a-z0-9 ]", '', name)  # remove punctuation
    name = re.sub(r'\s+', ' ', name)          # collapse spaces
    return name

try:
    steam = pd.read_csv('steam_games.csv', usecols=['name', 'popular_tags'])
    steam = steam.dropna(subset=['name', 'popular_tags'])
    steam['norm_name'] = steam['name'].apply(normalize_name)
    steam = steam.drop_duplicates('norm_name')

    # Build lookup dict: normalized_name -> tags string
    tag_lookup = dict(zip(steam['norm_name'], steam['popular_tags']))

    df['norm_name'] = df['game_name'].apply(normalize_name)
    df['tags'] = df['norm_name'].map(tag_lookup).fillna('No tags found')

    matched = (df['tags'] != 'No tags found').sum()
    print(f'Tags merged! {matched}/{len(df)} games matched ({matched*100//len(df)}%)')
    print('\nSample tags:')
    print(df[df['tags'] != 'No tags found'][['game_name','tags']].head(10).to_string())

except FileNotFoundError:
    print('steam_games.csv not found — using placeholder tags for now.')
    print('Upload steam_games.csv from Kaggle to get real tags.')
    df['tags'] = df['genre'].apply(
        lambda g: re.sub(r"[\[\]']", '', str(g)).replace(',', ',').strip()
    )


### Step 3 — Rebuild TF-IDF Including Tags
Tags are added to `combined_features` so prompts like 'souls-like' or 'open world' match tag words directly.


In [ ]:
def clean_list_string(text):
    if pd.isna(text): return ''
    return re.sub(r',', ' ', re.sub(r"[\[\]'\"]", '', str(text))).strip().lower()

def clean_text(text):
    return '' if pd.isna(text) else str(text).strip().lower()

df['clean_name']     = df['game_name'].apply(clean_text)
df['clean_genre']    = df['genre'].apply(clean_list_string)
df['clean_platform'] = df['platform'].apply(clean_list_string)
df['clean_type']     = df['type'].apply(clean_text)
df['clean_desc']     = df['description'].apply(clean_text)
df['clean_tags']     = df['tags'].apply(clean_list_string)

df['combined_features'] = (
    df['clean_name']  + ' ' + df['clean_name']  + ' ' + df['clean_name']  + ' ' + # name x3
    df['clean_desc']  + ' ' +
    df['clean_genre'] + ' ' + df['clean_genre'] + ' ' +                            # genre x2
    df['clean_tags']  + ' ' + df['clean_tags']  + ' ' +                            # tags x2
    df['clean_platform'] + ' ' +
    df['clean_type']  + ' ' + df['clean_type']
)

print('Rebuilding TF-IDF with tags included...')
tfidf = TfidfVectorizer(
    max_features=15000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf.fit_transform(df['combined_features'])
print(f'TF-IDF rebuilt. Matrix: {tfidf_matrix.shape[0]} games x {tfidf_matrix.shape[1]} features')


### Step 4 — Keyword Expansion + Name Matching

In [ ]:
KEYWORD_MAP = {
    'hardcore':       'souls-like unrelenting challenge brutal difficult demanding combat death',
    'hard':           'unrelenting challenge difficult brutal demanding',
    'souls like':     'souls-like dark souls unrelenting challenge combat bosses death action rpg',
    'soulslike':      'souls-like dark souls unrelenting challenge combat bosses death action rpg',
    'soulsborne':     'souls-like dark souls bloodborne unrelenting challenge combat bosses death',
    'challenging':    'unrelenting challenge brutal difficult demanding souls-like',
    'punishing':      'unrelenting challenge brutal death difficult souls-like',
    'fps':            'first-person shooter first person',
    'tps':            'third-person shooter third person',
    'battle royale':  'battle royale survival competitive multiplayer last standing',
    'roguelike':      'roguelike procedural randomly generated permadeath run',
    'roguelite':      'roguelite procedural permadeath run progression',
    'metroidvania':   'metroidvania exploration platformer ability unlock backtrack',
    'sandbox':        'open world sandbox freedom build create explore',
    'stealth':        'stealth sneaking hide detection silent',
    'horror':         'horror survival fear dark twisted',
    'chill':          'relaxing casual peaceful slow paced family friendly',
    'story driven':   'story rich narrative compelling deep plot characters',
    'co-op':          'co-op cooperative multiplayer teamwork',
    'coop':           'co-op cooperative multiplayer teamwork',
    'loot':           'loot items weapons gear equipment drops',
    'hack slash':     'hack and slash combat action fast melee',
    'turn based':     'turn-based strategy tactical',
    'city builder':   'city builder building management simulation construction',
    'tower defense':  'tower defense strategy waves enemies',
    'jrpg':           'jrpg japanese role playing story characters turn',
    'moba':           'moba multiplayer battle arena team strategy',
    'rts':            'rts real time strategy command troops resource',
    'open world':     'open world exploration vast sandbox freedom',
    'survival':       'survival crafting base building resources hunger',
    'platformer':     'platformer jump run side-scrolling 2d 3d platform',
}

def expand_prompt(prompt):
    expanded = prompt.lower()
    for term, expansion in KEYWORD_MAP.items():
        if term in expanded:
            expanded += ' ' + expansion
    return expanded

COMMON_WORDS = {'game','with','from','that','this','open','world','good',
                'great','best','like','play','want','need','find','show','more'}

def find_name_matches(prompt):
    p = prompt.lower().strip()
    exact = df[df['clean_name'].str.contains(re.escape(p), na=False)].index.tolist()
    if exact: return exact[:6]
    words = p.split()
    if len(words) >= 2:
        for i in range(len(words) - 1):
            phrase = words[i] + ' ' + words[i+1]
            if len(phrase) > 6:
                hits = df[df['clean_name'].str.contains(re.escape(phrase), na=False)].index.tolist()
                if hits: return hits[:6]
    sig = [w for w in words if len(w) > 4 and w not in COMMON_WORDS]
    if len(sig) == 1:
        hits = df[df['clean_name'].str.contains(
            r'\b' + re.escape(sig[0]) + r'\b', na=False, regex=True
        )].index.tolist()
        return hits[:6]
    return []

print('Keyword map and name matcher ready!')


### Step 5 — Final Hybrid Recommender (20 results + tags)

In [ ]:
def recommend_games(user_prompt, top_n=20, alpha=0.7, beta=0.3):
    """
    Final hybrid recommender.
    Returns top_n=20 games with tags column included.

    Parameters:
        user_prompt : str  — natural language description
        top_n       : int  — number of results (default 20)
        alpha       : float — weight for similarity  (default 0.7)
        beta        : float — weight for sentiment   (default 0.3)
    """

    expanded = expand_prompt(user_prompt)
    user_vector = tfidf.transform([expanded])
    sim_scores  = cosine_similarity(user_vector, tfidf_matrix).flatten()
    sentiment   = df['sentiment_score'].values

    # Normalize both to [0,1] so sentiment never drowns relevance
    max_sim  = sim_scores.max()
    norm_sim  = (sim_scores / max_sim) if max_sim > 0 else sim_scores
    norm_sent = sentiment / sentiment.max()

    final_scores = alpha * norm_sim + beta * norm_sent

    # Dynamic threshold: game must be >=20% as relevant as best match
    min_sim = max(0.04, max_sim * 0.20)
    final_scores[sim_scores < min_sim] = -1.0

    # Pin name matches to top
    name_hits = find_name_matches(user_prompt)
    for idx in name_hits:
        final_scores[idx] = 2.0 + sim_scores[idx]

    top_idx = [i for i in final_scores.argsort()[::-1] if final_scores[i] > 0][:top_n]

    if not top_idx:
        print('No strong matches found. Try different keywords.')
        return pd.DataFrame()

    results = df.iloc[top_idx][[
        'game_name', 'genre', 'type', 'meta_score', 'user_score', 'rating', 'tags'
    ]].copy()

    results['similarity']  = sim_scores[top_idx].round(4)
    results['sentiment']   = sentiment[top_idx].round(4)
    results['final_score'] = final_scores[top_idx].round(4)
    results.index = range(1, len(top_idx)+1)
    return results

print('recommend_games() is ready! Default top_n=20 with tags column.')


### Step 6 — Test It

In [ ]:
prompt = 'dark story driven RPG with open world exploration'
print(f"Prompt: '{prompt}'")
print('='*70)
r = recommend_games(prompt, top_n=20)
# Show key columns including tags
print(r[['game_name', 'meta_score', 'similarity', 'final_score', 'tags']].to_string())


In [ ]:
# Name search test
prompt = 'dark souls'
print(f"Prompt: '{prompt}'")
print('='*70)
r = recommend_games(prompt, top_n=20)
print(r[['game_name', 'meta_score', 'similarity', 'final_score', 'tags']].to_string())


In [ ]:
# Interactive mode — now returns 20 results with tags
while True:
    prompt = input('\nDescribe the game you want (or type quit): ')
    if prompt.lower() == 'quit':
        print('Done.')
        break
    if len(prompt.strip()) < 3:
        print('Please type a longer description.')
        continue
    res = recommend_games(prompt, top_n=20)
    if len(res) == 0:
        print('No matches. Try different keywords.')
    else:
        print(f"\nTop {len(res)} games for: '{prompt}'")
        print('-'*70)
        print(res[['game_name', 'meta_score', 'similarity', 'final_score', 'tags']].to_string())


## Phase 3 Complete ✅

| Feature | Detail |
|---------|--------|
| Results | 20 per query (up from 10) |
| Tags column | Merged from `steam_games.csv` (Kaggle) by game name |
| Tags in TF-IDF | Tags added to `combined_features` with 2x weight |
| Keyword expansion | 30+ gaming terms mapped to description vocabulary |
| Name matching | Exact/phrase match pins named games to top |
| Normalized scoring | Sentiment never dominates irrelevant games |

**Note:** Elden Ring is not in this dataset — it postdates the dataset collection.

**Next → Phase 4:** Streamlit UI.